In [ ]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.gaia import Gaia

In [ ]:
Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"

coord = SkyCoord(ra='291.3656402942d', dec='42.7834887926d')  # rr lyrae

# Set the search parameters
search_radius = 25 * u.arcmin  # Replace with your desired radius
mag_limit = 15  # Replace with your desired magnitude limit

In [ ]:
query = f"""
SELECT TOP 1000
    source_id, ra, dec, phot_g_mean_mag,
    phot_bp_mean_mag, phot_rp_mean_mag, 
    bp_rp, bp_g, g_rp, -- Precomputed color indices
    teff_gspphot,  -- Effective temperature
    phot_variable_flag,
    classprob_dsc_combmod_star
FROM gaiadr3.gaia_source
WHERE CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', {coord.ra.deg}, {coord.dec.deg}, {search_radius.to(u.deg).value})
) = 1
AND phot_variable_flag != 'VARIABLE'
AND phot_g_mean_mag < {mag_limit}
AND (classprob_dsc_combmod_star > 0.9) -- Probability it's a star
"""

# Execute the query
job = Gaia.launch_job(query)
gaia_table = job.get_results()

In [ ]:
np.sum(gaia_table['phot_variable_flag'] == 'NOT_AVAILABLE')

In [ ]:
np.sum(gaia_table['phot_variable_flag'] == 'VARIABLE')

In [ ]:
len(gaia_table) == np.sum(gaia_table['phot_variable_flag'] == 'NOT_AVAILABLE') + np.sum(gaia_table['phot_variable_flag'] == 'VARIABLE')

In [ ]:
len(gaia_table)

In [ ]:
59+ 862